# Trilingual NMT — Italian bridge (NLLB-1.3B)

Add **Italian** (`ita_Latn`) as a bridge language. Steps:
1. Evaluate NLLB-1.3B **zero-shot** on all directions: scn↔en and **it↔scn**.
2. **Multilingual fine-tune** (LoRA) on four directions jointly: scn→en, en→scn,
   it→scn, scn→it.
3. Re-evaluate all directions on the frozen test sets.

Data on Drive (`SicilianNMT-colab/data/`): train/valid/test.{scn,en} (scn-en) and
itsc_{train,valid,test}.{it,scn} (frozen WikiMatrix it-scn split). Best so far
(scn→en): bidirectional LoRA 31.33 BLEU.

In [ ]:
!pip -q install transformers datasets peft sentencepiece sacrebleu accelerate
!pip -q uninstall -y torchao

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/SicilianNMT-colab'
except Exception:
    OUT = 'sicilian-nllb-out'
os.makedirs(OUT, exist_ok=True)
print('using', OUT)

In [ ]:
def read(p): return open(p, encoding='utf-8').read().splitlines()
DATA = f'{OUT}/data'
base = DATA if os.path.exists(f'{DATA}/train.scn') else '.'
print('reading data from', base)
se = {s: {l: read(f'{base}/{s}.{l}') for l in ('scn', 'en')} for s in ('train', 'valid', 'test')}
isc = {s: {l: read(f'{base}/itsc_{s}.{l}') for l in ('it', 'scn')} for s in ('train', 'valid', 'test')}
print('scn-en:', {s: len(d['scn']) for s, d in se.items()})
print('it-scn:', {s: len(d['it']) for s, d in isc.items()})

In [ ]:
import os, gc, json, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from sacrebleu.metrics import BLEU, CHRF

for _v in ('model', 'ft'):
    if _v in globals():
        del globals()[_v]
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()

MODEL = 'facebook/nllb-200-1.3B'
LANG = {'scn': 'scn_Latn', 'en': 'eng_Latn', 'it': 'ita_Latn'}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32
tok = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL, torch_dtype=dtype, low_cpu_mem_usage=True).to(device).eval()
results = {}

def translate(texts, src, tgt, bs=8, max_len=160):
    tok.src_lang = LANG[src]
    tgt_id = tok.convert_tokens_to_ids(LANG[tgt])
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i+bs], return_tensors='pt', padding=True, truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(**enc, forced_bos_token_id=tgt_id, max_length=max_len, num_beams=5)
        out += tok.batch_decode(gen, skip_special_tokens=True)
        print(f'  {min(i+bs, len(texts))}/{len(texts)}', end='\r')
    print(); torch.cuda.empty_cache()
    return out

def report(hyp, ref, tag):
    b = BLEU().corpus_score(hyp, [ref]).score
    c = CHRF().corpus_score(hyp, [ref]).score
    results[tag] = {'bleu': round(b, 2), 'chrf': round(c, 2)}
    json.dump(results, open(f'{OUT}/results_multi.json', 'w'), indent=2, ensure_ascii=False)
    print(f'{tag}:  BLEU={b:.2f}  chrF={c:.2f}')
print('ready on', device, '|', dtype)

In [ ]:
# ---- ZERO-SHOT, all directions ----
report(translate(se['test']['scn'], 'scn', 'en'),  se['test']['en'],  'zs scn->en')
report(translate(se['test']['en'],  'en', 'scn'),  se['test']['scn'], 'zs en->scn')
report(translate(isc['test']['it'], 'it', 'scn'),  isc['test']['scn'],'zs it->scn')
report(translate(isc['test']['scn'],'scn', 'it'),  isc['test']['it'], 'zs scn->it')

## Multilingual LoRA fine-tune (scn↔en + it↔scn)

In [ ]:
from datasets import Dataset, concatenate_datasets
from peft import LoraConfig, get_peft_model
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

torch.cuda.empty_cache()
EPOCHS = 1   # four directions => a lot of data; raise if time allows

ft = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.05, bias='none',
                                      target_modules=['q_proj', 'k_proj', 'v_proj', 'out_proj'],
                                      task_type='SEQ_2_SEQ_LM'))
for p in ft.parameters():
    if p.requires_grad:
        p.data = p.data.float()
ft.config.use_cache = False
ft.enable_input_require_grads()
ft.print_trainable_parameters()

def tok_dir(src_list, tgt_list, src_lang, tgt_lang):
    tok.src_lang, tok.tgt_lang = src_lang, tgt_lang
    return Dataset.from_dict({'src': src_list, 'tgt': tgt_list}).map(
        lambda b: tok(b['src'], text_target=b['tgt'], max_length=128, truncation=True),
        batched=True, remove_columns=['src', 'tgt'])

parts = [
    tok_dir(se['train']['scn'], se['train']['en'], 'scn_Latn', 'eng_Latn'),
    tok_dir(se['train']['en'], se['train']['scn'], 'eng_Latn', 'scn_Latn'),
    tok_dir(isc['train']['it'], isc['train']['scn'], 'ita_Latn', 'scn_Latn'),
    tok_dir(isc['train']['scn'], isc['train']['it'], 'scn_Latn', 'ita_Latn'),
]
train_ds = concatenate_datasets(parts).shuffle(seed=13)
print('multilingual training examples:', len(train_ds))

args = Seq2SeqTrainingArguments(
    output_dir=f'{OUT}/trainer-multi-1.3B', num_train_epochs=EPOCHS,
    per_device_train_batch_size=4, gradient_accumulation_steps=4,
    gradient_checkpointing=True, gradient_checkpointing_kwargs={'use_reentrant': False},
    learning_rate=2e-4, fp16=True, logging_steps=100, save_strategy='no', report_to=[])
Seq2SeqTrainer(model=ft, args=args, train_dataset=train_ds,
               data_collator=DataCollatorForSeq2Seq(tok, model=ft)).train()

In [ ]:
# ---- EVALUATE FINE-TUNED, all directions + SAVE ADAPTER ----
model = ft.eval(); model.config.use_cache = True
report(translate(se['test']['scn'], 'scn', 'en'),  se['test']['en'],  'ft scn->en')
report(translate(se['test']['en'],  'en', 'scn'),  se['test']['scn'], 'ft en->scn')
report(translate(isc['test']['it'], 'it', 'scn'),  isc['test']['scn'],'ft it->scn')
report(translate(isc['test']['scn'],'scn', 'it'),  isc['test']['it'], 'ft scn->it')
ft.save_pretrained(f'{OUT}/nllb-lora-multi-1.3B')
print('saved adapter + results_multi.json to', OUT)
print(json.dumps(results, indent=2, ensure_ascii=False))